In [28]:
import requests
from bs4 import BeautifulSoup

url="https://www.gutenberg.org/cache/epub/84/pg84-images.html"
response=requests.get(url)
soup=BeautifulSoup(response.text, "html.parser")

text=soup.get_text()
text=" ".join(text.split())
print(text[:500])

Frankenstein | Project Gutenberg The Project Gutenberg eBook of Frankenstein; or, the modern prometheus This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org. If you are not located in the United States, you will have to check the laws of the country 


In [29]:
import tensorflow as tf

tokenizer=tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts([text])

sequence=tokenizer.texts_to_sequences([text])[0]

vocab_size=len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 7632


In [30]:
import numpy as np

window_size=100
inputs=[]
targets=[]

for i in range(len(sequence) - window_size):
    inputs.append(sequence[i:i+99])
    targets.append(sequence[i+99])

inputs=np.array(inputs)
targets=np.array(targets)

print(inputs.shape)
print(targets.shape)

(78580, 99)
(78580,)


In [41]:
model=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(99,)),
    tf.keras.layers.Embedding(vocab_size, 64),
    tf.keras.layers.SimpleRNN(128),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 99, 64)         │       488,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_7 (SimpleRNN)        │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 7632)           │       984,528 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,497,680 (5.71 MB)

 Trainable params: 1,497,680 (5.71 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
model.fit(inputs, targets, epochs=40, batch_size=64)

Epoch 1/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - accuracy: 0.0501 - loss: 7.1525
Epoch 2/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.0751 - loss: 6.4006
Epoch 3/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1038 - loss: 6.1058
Epoch 4/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1263 - loss: 5.8667
Epoch 5/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1194 - loss: 6.1362
Epoch 6/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1442 - loss: 5.5547
Epoch 7/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1530 - loss: 5.3557
Epoch 8/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1650 - loss: 5.1922
Epoch 9/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1713 - loss: 5.1185
Epoch 10/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1772 - loss: 5.0544
Epoch 11/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.1903 - loss: 4.7415
Epoch 12

In [43]:
import numpy as np

def generate(seed_text, num_words=50):
    for _ in range(num_words):
        token_list=tokenizer.texts_to_sequences([seed_text])[0]
        token_list=token_list[-99:]
        token_list=tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=99, padding='pre'
        )

        prediction=model.predict(token_list, verbose=0)
        predicted_word_index=np.argmax(prediction)

        index_word=tokenizer.index_word
        seed_text += " " + index_word.get(predicted_word_index, "")

    return seed_text

In [44]:
print(generate("the monster was very miserable and wandered through the forest", 100))

the monster was very miserable and wandered through the forest this and the more days of my own species it is a few years i shed the greatest eagerness to the horror of the murder i replied the boat and was much on this i was already unacquainted with the horror of my father and with a slight air and the masters of the judges did you were nearly to him borne down as well as heat and that i might have heard him for the object of my native country i did not participate in the harbour ignorant of nature i had no compass with me to conceal my


# **LSTM**

In [35]:
model_2=tf.keras.Sequential([
    tf.keras.layers.Input(shape=(99,)),
    tf.keras.layers.Embedding(vocab_size, 128),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.LSTM(256),
    tf.keras.layers.Dense(vocab_size, activation="softmax")
])

model_2.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model_2.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 99, 128)        │       976,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 99, 256)        │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 7632)           │     1,961,424 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,857,872 (14.72 MB)

 Trainable params: 3,857,872 (14.72 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
model_2.fit(inputs, targets, epochs=40, batch_size=64)

Epoch 1/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 29s 22ms/step - accuracy: 0.0596 - loss: 6.9092
Epoch 2/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.0868 - loss: 6.1689
Epoch 3/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.1240 - loss: 5.7334
Epoch 4/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.1396 - loss: 5.4180
Epoch 5/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.1524 - loss: 5.1314
Epoch 6/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.1616 - loss: 4.8616
Epoch 7/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.1737 - loss: 4.5780
Epoch 8/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.1900 - loss: 4.3248
Epoch 9/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 26s 22ms/step - accuracy: 0.2062 - loss: 4.0635
Epoch 10/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 41s 22ms/step - accuracy: 0.2332 - loss: 3.8078
Epoch 11/40
1228/1228 ━━━━━━━━━━━━━━━━━━━━ 27s 22ms/step - accuracy: 0.2634 - loss: 3.5865
Epoch 12

In [37]:
import numpy as np

def generate2(seed_text, num_words=50):
    for _ in range(num_words):
        token_list=tokenizer.texts_to_sequences([seed_text])[0]
        token_list=token_list[-99:]
        token_list=tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=99, padding='pre'
        )

        prediction=model_2.predict(token_list, verbose=0)
        predicted_word_index=np.argmax(prediction)

        index_word=tokenizer.index_word
        seed_text += " " + index_word.get(predicted_word_index, "")

    return seed_text

In [38]:
print(generate2("the monster was very miserable and wandered through the forest", 100))

the monster was very miserable and wandered through the forest however where hills which country our food i took free from his face free muscles and fears with attention to divine sensations or firm in the most ground free de blanc in waters and was solely wrapped up in connected to the cottagers early and in its inhabitants and they also arrived to find a few minutes on the cottage and on its 11th i contemplated with the little recurrence and marked that i had never beheld his tools in a boat and dogs were always hid in a thick cadence swelling were covered for the means of obtaining the


# Եզրակացություն

Այս աշխատանքում տեքստի գեներացման համար կիրառվել են երկու տարբեր մոդելներ, SimpleRNN և LSTM։ Որպես ուսուցման տվյալներ օգտագործվել է Frankenstein գրքի տեքստը, որը վերցվել է Project Gutenberg կայքից։

SimpleRNN մոդելը 40 epoch-ից հետո ցույց տվեց մոտ 0.71 accuracy և 1.30 loss։ Սա ցույց է տալիս, որ մոդելը սովորեց բառերի որոշ կապեր և նախադասությունների կառուցվածքի որոշ օրինաչափություններ։ Սակայն գեներացված տեքստում նկատվում է իմաստային խզում։ Քանի որ օգտագործված էր սովորական RNN այդպիսի արդյունքը սպասելի էր։ RNN-ները դժվարանում են երկար հաջորդականությունների դեպքում պահպանել նախորդ կոնտեքստը և հաճախ մոռանում են ավելի վաղ բառերը։

LSTM մոդելը ցույց տվեց զգալիորեն ավելի լավ արդյունքներ մոտ 0.94 accuracy և 0.37 loss։ Այն ավելի լավ է հասկանում, թե որ բառերն են կապված միմյանց հետ նույնիսկ ավելի երկար կոնտեքստում։

Ընդհանուր առմամբ, արդյունքները ցույց են տալիս, որ LSTM մոդելը ավելի արդյունավետ է, քան SimpleRNN-ը, երբ խնդիրը կապված է երկար տեքստային հաջորդականությունների մշակման և հաջորդ բառի կանխատեսման հետ։